<a href="https://colab.research.google.com/github/Edomario082909/AHHHHR/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import threading, subprocess, re

# =========================
# 🔒 GESTIONE TOKEN
# =========================
MY_CIVITAI_TOKEN = "a84327c4d857915fd6730f0f0b637aac"

# =========================
# 🔧 SETUP SISTEMA
# =========================
!rm -f /etc/apt/sources.list.d/r2u.list
!apt -y update -qq && apt -y install -qq aria2 ffmpeg libsox-dev
!wget -q https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

# =========================
# 📦 INSTALLAZIONE COMFYUI
# =========================
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI

# Installazione dipendenze base
!pip install -q torchsde omegaconf diffusers accelerate
!pip install -q -r requirements.txt

# ==========================================
# 🎵 AUDIO & EXTRA
# ==========================================
!pip install -q lucent taming-transformers-rom1504
!rm -rf /content/ComfyUI/comfy_api_nodes

# =========================
# 🧩 CUSTOM NODES
# =========================
%cd /content/ComfyUI/custom_nodes

!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
!pip install -q gguf
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

# =========================
# 📥 DOWNLOAD MODELLI
# =========================
%cd /content/ComfyUI

my_list = [
    ("2747549", "ltx_lora_fov.safetensors", "loras"),
    ("2754108", "ltx_lora_2.safetensors", "loras"),
    ("2617751", "zimage_lora_1.safetensors", "loras"),
    ("2442439", "zimage_checkpoint.safetensors", "checkpoints"),
    ("2465980", "zimage_lora_2.safetensors", "loras"),
    ("2474435", "zimage_lora_3.safetensors", "loras"),
    ("2083303", "wan_lora_1.safetensors", "loras"),
    ("2200388", "wan_lora_2.safetensors", "loras"),
    ("2475163", "zimage_lora_4.safetensors", "loras"),
    ("2477457", "zimage_lora_5.safetensors", "loras"),
    ("2581135", "zimage_lora_6.safetensors", "loras"),
    ("2273468", "wan_lora_3.safetensors", "loras"),
    ("2441730", "wan_lora_4.safetensors", "loras"),
    ("2553151", "wan_lora_5.safetensors", "loras"),
    ("2460437", "zimage_lora_7.safetensors", "loras"),
    ("2073605", "wan_lora_6.safetensors", "loras"),
    ("2739037", "sdxl_real_checkpoint.safetensors", "checkpoints"),
    ("1811054", "sdxl_lora_1.safetensors", "loras"),
    ("2752410", "ltx_checkpoint.safetensors", "checkpoints"),
    ("2376136", "wan_lora_7.safetensors", "loras"),
    ("2342652", "wan_lora_8.safetensors", "loras"),
    ("2176450", "wan_lora_9.safetensors", "loras"),
    ("2430424", "wan_lora_10.safetensors", "loras"),
    ("2738377", "grok_checkpoint.safetensors", "checkpoints")
]

def download_model(v_id, name, folder):
    # Torniamo al metodo funzionante col token integrato nell'URL
    url = f"https://civitai.com/api/download/models/{v_id}?token={MY_CIVITAI_TOKEN}"
    print(f"⬇️ In download: {name}")

    cmd = f'aria2c -c -x 16 -s 16 -k 1M "{url}" -d /content/ComfyUI/models/{folder} -o {name}'
    exit_code = os.system(cmd)

    if exit_code == 0:
        print(f"✅ OK: {name}")
    else:
        print(f"❌ Errore/Skip: {name}")
        # Puliamo i file vuoti o corrotti se il download fallisce
        os.system(f'rm -f /content/ComfyUI/models/{folder}/{name} /content/ComfyUI/models/{folder}/{name}.aria2')

for v_id, name, folder in my_list:
    download_model(v_id, name, folder)

# =========================
# 🌐 TUNNEL CLOUDFLARE
# =========================
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64
!chmod 777 /content/cloudflared-linux-amd64

def tunnel():
    p = subprocess.Popen(
        ['/content/cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    for line in iter(p.stdout.readline, b''):
        line = line.decode()
        link = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if link:
            print(f"\n🟢 LINK COMFYUI: {link.group(0)}\n")

threading.Thread(target=tunnel, daemon=True).start()

# =========================
# ▶️ AVVIO
# =========================
!python main.py --dont-print-server --listen 0.0.0.0
